# Spectrometer Calibration Notebook (Modified from Knife edge Scan)

In [ ]:
# Import python libs and utils.py
import numpy as np 
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import special
import xrayscatteringtools as xrst
xrst.enable_underscore_cleanup()

### Load the data in

In [ ]:
###############################################
runNumbers = []  # Enter the run numbers to be loaded
folders = xrst.get_data_paths(runNumbers) # Defaults to the info in config.yaml. You can overwrite this with strings, character arrays, or lists of either.
###############################################
# (1) keys_to_combine: some keys loaded for each shot & stored per shot 
# (2) keys_to_sum: some keys loaded per each run and added 
# (3) keys_to_check : check if some keys exits and have same values in all runs and load these keys 
_keys_to_combine = [
    # Monochromator Information
    # 'epicsUser/SP1L0_theta_1', # This may have to be replaced with the daq scan equivalents of these variables if we are scanning them
    # 'epicsUser/SP1L0_theta_2',
    # 'scan/SP1L0_theta_1',
    'scan/DCCM_energy',
    # Spectrometer Linout
    'feeBld/hproj',
    # Gas Detectors
    'gas_detector/f_11_ENRC', 'gas_detector/f_22_ENRC',
    # Event Codes
    'lightStatus/laser',
    'lightStatus/xray'
]
_keys_to_sum = [] # None here
_keys_to_check = [] # None here
# Load the data in
_data = xrst.combineRuns(runNumbers, folders, _keys_to_combine, _keys_to_sum, _keys_to_check, verbose=False)  # this is the function to load the data with defined keys

# -- Turn the DCCM Mono angle into energy.
_a = 5.42941059472 # Si Lattice Constant at 77 K in angstroms, from Shah et. al., 1972
_reflection = np.asarray([1,1,1]) # Si 111 reflection
_d = _a / np.sqrt(np.sum(_reflection**2)) # Interplanar spacing
# Bragg formula, convert to keV
# mono_keV = xrst.Angstroms2keV(2 * _d * np.sin(_data['epicsUser/SP1L0_theta_1']*np.pi/180))
mono_keV = 'scan/DCCM_energy'
# -- Spectrometer lineout
spec = _data['feeBld/hproj']
# -- Pulse Intensity
pulse_energy = _data['gas_detector/f_22_ENRC'] # pulse energy (likely in mJ)
# -- Event Codes / Shot Status
laserOn = _data['lightStatus/laser'].astype(bool)  # laser on events
xrayOn = _data['lightStatus/xray'].astype(bool)  # xray on events
run_indicator = _data['run_indicator'] # run indicator for each shot

In [ ]:
########## Different filter cutoffs##########
_gdet_cutoff = [0.03, 3];
_spec_sum_cutoff = [0, 1];
#############################################

# Precomputing the normalized values
_specNorm = np.nansum(spec,axis=1)/np.nansum(spec,axis=1).max();


plt.figure(figsize=[17,5]) 
plt.subplot(1,2,1)
plt.hist(pulse_energy,bins=200);
plt.axvline(_gdet_cutoff[0],color='r',linestyle='--')
plt.axvline(_gdet_cutoff[1],color='r',linestyle='--')
plt.axvspan(_gdet_cutoff[0],_gdet_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale('log')
plt.title('Gas Detector Histogram')
plt.xlabel('Pulse Energy (mJ)')
plt.ylabel('Counts')
plt.legend()

plt.subplot(1,2,2)
plt.hist(_specNorm,bins=200);
plt.axvline(_spec_sum_cutoff[0],color='r',linestyle='--')
plt.axvline(_spec_sum_cutoff[1],color='r',linestyle='--')
plt.axvspan(_spec_sum_cutoff[0],_spec_sum_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale('log')
plt.title('Spectrometer Sum Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')
plt.legend()

plt.show()
goodIdx = np.logical_and.reduce([
    xrayOn,
    _gdet_cutoff[0] <= pulse_energy,
    pulse_energy <= _gdet_cutoff[1],
    _spec_sum_cutoff[0] <= _specNorm,
    _specNorm <= _spec_sum_cutoff[1]
])

# Displaying how much data was kept due to this filtering
_counts,_bins = np.histogram(goodIdx.astype(int),[0,1,2]);
print(f'Good data represents {_counts[1]/np.sum(_counts)*100:.2f}% of the total shots. ({_counts[1]} out of {np.sum(_counts)}).')

# Bin the shots, determine peak centers
As a note, this may need to be edited if the mono spits out noisy data for the angle position. We may have to define bins and then bin the shots relative to that with a digitize or bincount. I will figure that out later once I have data to try this out on.

In [ ]:
#########################
# Scan variable (dependent)
scan = mono_keV
# Aggregate data by unique values
_scanGood = scan[goodIdx]
_specGood = spec[goodIdx]
scan_unique = np.unique(_scanGood) 
pixel_mean = []
pixel_std = []
# Sort the data by the scan value, take the mean and std
def custom_gaussian(x, a, sigma, mu, b):
    return a * np.exp(-((x - mu)**2) / (2 * sigma**2)) + b

# Do the calculation and the binning
_spec_pixel_array = np.arange(spec.shape[1])
for value in scan_unique:
    # Extract out the spectra from the scan point
    _mask = _scanGood == value
    _mean_spec_norm = np.nanmean(_specGood[_mask],axis=0)/np.nanmean(_specGood[_mask],axis=0).max()
    # Fit the mean to a gaussian
    _popt, _pcov = curve_fit(custom_gaussian,_spec_pixel_array, _mean_spec_norm,p0=(1, 1,np.nanargmax(_mean_spec_norm),0))
    # Extract out the center position and then figure out the uncertainty in the center.
    # plt.plot(_spec_pixel_array,_mean_spec_norm) # <- For debugging
    pixel_mean.append(_popt[2])
    pixel_std.append(np.sqrt(_pcov[2, 2]))

## Fit the data

In [ ]:
#######################
poly_order = 1 # This can be changed and the rest of the notebooks will adapt to the different orders
#######################

# Fit the final data to a polynomial
coefficients = np.polyfit(scan_unique, pixel_mean, poly_order)
_poly_func = np.poly1d(coefficients)

# Generate fitted data for plotting
_x_fit = np.linspace(scan_unique.min(), scan_unique.max(), 500)
_y_fit = _poly_func(_x_fit)

# Plot it
plt.figure(figsize=(6, 5))
plt.errorbar(scan_unique, pixel_mean, yerr=pixel_std, fmt='o', markersize=4, 
             capsize=3, ecolor='gray', alpha=0.7, label='Data w/ Uncertainty')
plt.plot(_x_fit, _y_fit, label=f'Poly Fit (Order {poly_order})', color='red', linewidth=2)

plt.xlabel('DCCM Photon Energy (keV)')
plt.ylabel('XRT Spectrometer Center Pixel Position')
plt.title('XRT Spectrometer Calibration')
plt.legend()
plt.show()

# Print fitted parameters
print(f"Fitted Polynomial Coefficients (Order {poly_order}):")
print('Save these to the config file for applying to the ghost imaging notebook')
print(coefficients)